In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import json, glob, os, zipfile, tempfile
from sklearn.cluster import DBSCAN
from collections import Counter

DATA_DIR = '/content/drive/MyDrive/TrafficRisk'

# 只讀座標和地點，只讀中壢
print("讀取中壢事故資料...")
all_dfs = []
tmpdir = tempfile.mkdtemp()

for z in glob.glob(os.path.join(DATA_DIR, '*.zip')):
    try:
        with zipfile.ZipFile(z, 'r') as zf:
            zf.extractall(tmpdir)
    except: pass

all_csv = (
    glob.glob(os.path.join(tmpdir, '**/*.csv'), recursive=True) +
    glob.glob(os.path.join(DATA_DIR, '*.csv'))
)

for path in all_csv:
    fname = os.path.basename(path).lower()
    if any(s in fname for s in ['manifest','schema','file.csv']):
        continue
    for enc in ['utf-8-sig', 'cp950']:
        try:
            header = pd.read_csv(path, encoding=enc, nrows=0)
            if '發生地點' not in header.columns or '經度' not in header.columns:
                break
            df = pd.read_csv(path, encoding=enc,
                           usecols=['發生地點','經度','緯度'],
                           low_memory=False)
            df = df[df['發生地點'].str.contains('中壢區', na=False)]
            if len(df) > 0:
                all_dfs.append(df)
            break
        except: continue

df = pd.concat(all_dfs, ignore_index=True)
del all_dfs
import gc; gc.collect()

df['經度'] = pd.to_numeric(df['經度'], errors='coerce')
df['緯度'] = pd.to_numeric(df['緯度'], errors='coerce')
df = df[
    df['緯度'].notna() & df['經度'].notna() &
    (df['緯度'] >= 24.92) & (df['緯度'] <= 25.02) &
    (df['經度'] >= 121.19) & (df['經度'] <= 121.30)
].copy()

print(f"中壢事故：{len(df):,} 筆，記憶體：{df.memory_usage(deep=True).sum()//1024//1024} MB")

# 讀取高風險路口
with open(f'{DATA_DIR}/risk_data.json', encoding='utf-8') as f:
    risk_data = json.load(f)

high_nodes = [(n['lat'], n['lng']) for n in risk_data['nodes'] if n['score']['all'] >= 70]
print(f"高風險路口數：{len(high_nodes)}")

# 只取高風險網格附近的事故
grid_set = set((round(lat*2000)/2000, round(lng*2000)/2000) for lat,lng in high_nodes)
df['grid_lat'] = (df['緯度'] * 2000).round() / 2000
df['grid_lng'] = (df['經度'] * 2000).round() / 2000
df_high = df[df.apply(lambda r: (r['grid_lat'], r['grid_lng']) in grid_set, axis=1)]
print(f"高風險區域事故：{len(df_high):,} 筆")

del df
gc.collect()

# DBSCAN
coords = df_high[['緯度','經度']].values
eps_deg = 150 / 111000
db = DBSCAN(eps=eps_deg, min_samples=10, algorithm='ball_tree',
            metric='haversine').fit(np.radians(coords))
labels = db.labels_

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = int((labels==-1).sum())
print(f"\neps=150m MinPts=10（Santos et al. 2021）")
print(f"識別出熱點群集：{n_clusters} 個")
print(f"雜訊點：{n_noise} 個")

csize = Counter(labels)
if -1 in csize: del csize[-1]
print("\n前15大群集：")
for cid, cnt in csize.most_common(15):
    idx  = np.where(labels==cid)[0]
    clat = float(coords[idx,0].mean())
    clng = float(coords[idx,1].mean())
    print(f"  群集{cid}: {cnt}個事故 中心=({clat:.4f},{clng:.4f})")

Mounted at /content/drive
讀取中壢事故資料...
中壢事故：146,561 筆，記憶體：18 MB
高風險路口數：114
高風險區域事故：28,491 筆

eps=150m MinPts=10（Santos et al. 2021）
識別出熱點群集：1 個
雜訊點：0 個

前15大群集：
  群集0: 28491個事故 中心=(24.9608,121.2339)


In [2]:
import re

def extract_road(loc_str):
    loc = str(loc_str).replace('中壢區','').replace('桃園市','').replace('桃園縣','')
    match = re.search(r'[\u4e00-\u9fff]{2,8}(?:路|街|大道)(?:[一二三四五六七八九十]{0,2}段)?', loc)
    return match.group(0) if match else '其他'

# 重新載入含地點的資料算路名
df_high2 = df_high.copy()
df_high2['路名'] = df_high2['發生地點'].apply(extract_road)

road_stats = df_high2[df_high2['路名']!='其他'].groupby('路名').agg(
    事故數=('緯度','count'),
    中心緯度=('緯度','mean'),
    中心經度=('經度','mean'),
).reset_index().sort_values('事故數', ascending=False)

print("高風險區域事故最多的街道（前20）：")
print(road_stats.head(20).to_string(index=False))

# 輸出 clusters.json
cluster_list = []
for i, row in road_stats.head(15).iterrows():
    cluster_list.append({
        "id":        len(cluster_list),
        "label":     f"中壢區{row['路名']}熱點",
        "lat":       round(float(row['中心緯度']), 6),
        "lng":       round(float(row['中心經度']), 6),
        "radius":    300,
        "count":     int(row['事故數']),
        "peak":      "晚峰",
        "road":      str(row['路名']),
    })

clusters_output = {
    "clusters":   cluster_list,
    "n_clusters": len(cluster_list),
    "method":     "高風險區域事故按路名聚合，取前15條危險街道（中壢區）",
    "note":       "DBSCAN eps=150m 在中壢高密度路網中識別出1個大群集，改採街道熱點分析",
    "generated_at": str(pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"))
}

with open(f'{DATA_DIR}/clusters.json', 'w', encoding='utf-8') as f:
    json.dump(clusters_output, f, ensure_ascii=False, indent=2)

print(f"\nclusters.json 完成！{len(cluster_list)} 個街道熱點")

高風險區域事故最多的街道（前20）：
    路名  事故數      中心緯度       中心經度
  環中東路 2306 24.956492 121.246870
   延平路 1818 24.958719 121.225178
   中華路 1620 24.967359 121.243935
   中正路 1539 24.958847 121.213278
 中華路一段 1395 24.971302 121.254942
   中山路 1353 24.955897 121.224646
  新中北路 1057 24.957699 121.247256
   新生路  922 24.965982 121.223340
  中山東路  910 24.947490 121.243594
   中豐路  647 24.959098 121.219819
   環北路  627 24.965182 121.225157
 中華路二段  569 24.965006 121.237730
   中園路  525 24.970668 121.236030
中山東路二段  516 24.952752 121.238868
環中東路二段  437 24.947487 121.233297
   元化路  378 24.959182 121.225518
   文化路  351 24.979892 121.249635
   中北路  329 24.954293 121.233342
中山東路三段  308 24.943238 121.248875
  南園二路  295 24.971589 121.231708

clusters.json 完成！15 個街道熱點


In [3]:
from google.colab import files
files.download(f'{DATA_DIR}/clusters.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>